# COMP0178 — Query Processing Part 2

Source: [Lecture 5: Query Processing Part 2](../Lectures/Query_Processing_2.pdf), 46 PDF pages. Page references below use the PDF page number. This continues the query-processing and optimization material from Lesson 4.

**Main idea:** an optimizer estimates the work needed by different query plans, then chooses a low-cost plan without exhaustively trying every possible plan. The estimates depend on table statistics, indexes, and the algorithms available.

The cost formulas here are simplified teaching models. A real DBMS can have more factors, and estimated cost is not an exact promise of elapsed time.


## 1. What the optimizer knows — statistics (PDF pages 4–9)

A database keeps summaries about tables and columns so it can estimate how many rows an operation will touch or return.

### Relation-wide statistics

- `nTuples(R)`: number of rows in relation `R` (cardinality).
- `bFactor(R)`: average number of tuples that fit in one storage block.
- `nBlocks(R)`: number of blocks needed to store/read `R`.

A basic estimate is:

```text
nBlocks(R) ≈ nTuples(R) / bFactor(R)
```

For example, 1,000 rows with 100 rows per block take about 10 blocks to scan.

### Attribute statistics

- `nDistinctA(R)`: number of distinct values of attribute `A` in `R`.
- `minA(R)` and `maxA(R)`: lowest and highest values.
- `SCA(R)`: estimated number of rows that satisfy a selection condition.

If values are assumed evenly distributed, an equality condition has the rough estimate:

```text
SCA(R, A = value) ≈ nTuples(R) / nDistinctA(R)
```

If there are 1,000 rows and 20 distinct department numbers, expect about 50 rows for one department. For a primary-key equality, the estimate is normally one row.

Range estimates use the min/max range; membership in a set can be estimated by multiplying the equality estimate by the number of requested values. These estimates can be wrong if the data is skewed. Statistics therefore need periodic updating; recomputing them after every update can itself be costly (page 7).


## 2. Indexes and hashing as access aids (PDF pages 8–9)

A full table scan checks rows one by one. An access aid can help the DBMS find matching rows with less work.

- An **index** maps indexed values to row locations. A primary key commonly has an index; a secondary index can be created on another attribute.
- A **clustered index** means the table rows are physically stored in the indexed order, so nearby key values tend to be stored near each other.
- A **multilevel index** organizes a large index as a tree. A **B+ tree** stays balanced, limiting the number of steps to reach a leaf.
- **Hashing** computes a bucket/location from a value. It can make equality lookup roughly constant-time, but collisions occur. A hash lookup is generally not the right tool for an ordered range such as `A < 50`.

An index is not automatically faster for every query. If a condition returns most of a table, scanning the table can be cheaper than many separate lookups.


## 3. Estimate a selection (σ) (PDF pages 11–14)

A selection filters rows. Its result size is the estimated selection cardinality, `SCA(R)`.

The access strategy depends on available ordering and search aids:

| Situation | Possible strategy |
|---|---|
| Unordered file, no index | Linear scan |
| Ordered file, no index | Binary search (for a suitable condition) |
| Hash available on the searched attribute | Hash lookup (especially equality) |
| Index available | Index lookup |

A **conjunction** (`condition1 AND condition2`) can test a useful, selective condition early so fewer rows need later checks. A composite index may help when conditions use multiple attributes. A **disjunction** (`OR`) must account for every branch; a slow branch can limit the whole operation.

Do not confuse result size with I/O cost: two plans can return the same rows but read different numbers of blocks.


## 4. Estimate a join and choose a join algorithm (PDF pages 15–20)

A join combines rows that satisfy a match condition. Its result cannot exceed the Cartesian product size:

```text
|R ⋈ S| ≤ nTuples(R) × nTuples(S)
```

In practice, the matching result is usually much smaller. Cost depends on relation sizes, available indexes/orderings, and the join algorithm.

### Main join strategies

1. **Basic nested-loop join:** for each row/block of `R`, compare it with rows/blocks of `S`. General-purpose, but can do a lot of repeated work.
2. **Indexed nested-loop join:** for each candidate row from the outer relation, use an index on the inner relation’s join key to find matches. Useful when that index exists and the outer side is reasonably small.
3. **Sort-merge join:** sort both relations on their join attributes, then scan the sorted inputs together. Suitable for equijoins; sorting has an upfront cost but can benefit later operations too.
4. **Hash join:** hash/partition both relations by the equijoin key, then match corresponding partitions. Often effective for equijoins when no useful index/order is available.

No single algorithm is always best. The optimizer compares what is available for this data and query.


## 5. Projection, set operations, and aggregation (PDF pages 21–24)

### Projection (π)

Projection keeps selected columns. It must read the input rows. If duplicate rows must be removed, the DBMS may sort or hash to find them. If the projected columns include a key, the output has as many rows as the input. If projecting one attribute, the result size is `nDistinctA(R)`.

Classical relational algebra treats relations as sets; SQL usually keeps duplicates unless `DISTINCT` is requested. This affects projection result counts.

### Set operations (∪, ∩, −)

The output size is bounded by the sizes of the input sets. Sorting or hashing can make it easier to match rows and eliminate duplicates.

### Aggregation

- An aggregate without `GROUP BY` returns one row; a scan can keep a running total/count/minimum/maximum.
- With grouping, output has about one row per distinct group key. The DBMS can sort or hash rows into groups, then scan each group to compute its aggregate.

The slide’s rough complexity shorthand uses `n log n` for sorting and `n` for hashing/scanning, where `n` is the amount of data being processed. This abstracts away block layout and implementation details.


## 6. Why the optimizer narrows its search (PDF pages 25–28)

Many equivalent plans can implement the same query. Even join order alone grows quickly: with three relations there are 12 join orderings in the lecture’s count; with `n` relations the count is `(2(n−1))! / (n−1)!`, before considering the different algorithms for each operation.

Trying every plan quickly becomes impractical. The optimizer therefore uses rules/restrictions and records good partial plans rather than starting over for every possibility.


## 7. Pipelining, materialization, and tree shapes (PDF pages 29–34)

- **Materialization:** compute an intermediate relation and write it to disk. A later operation must read it back.
- **Pipelining:** pass rows/blocks directly from one operation to the next without writing the full intermediate relation to disk.

A query tree describes logical operations; adding concrete algorithms (e.g. hash join) turns it into a physical **query evaluation plan** or **access plan** (page 30).

- **Bushy tree:** both sides of a join may be intermediate results; these may need to be materialized.
- **Linear tree:** one side is a base relation and the other may be an intermediate result.
- **Left-deep tree:** each new join adds one base relation on the right; the accumulated result stays on the left. This shape supports a simple pipeline.

The lecture’s search restrictions are: consider left-deep trees; pipeline selections/projections as early as possible; and avoid Cartesian products unless required by the query. These rules can omit a potentially faster plan, but greatly reduce planning work. Only join relations when a join predicate connects them.


## 8. Dynamic programming and interesting orders (PDF pages 35–38)

**Dynamic programming (DP)** solves a large optimization problem by remembering the best solutions to smaller subproblems and reusing them.

The lecture’s query-planning process is:

1. Find low-cost ways to access each single relation, applying available selections/projections early.
2. Use those results to find low-cost plans for pairs of relations.
3. Extend the best plans to three relations, then four, until all relations are included.
4. Ignore plans that require an unnecessary Cartesian product.

The **principle of optimality** is the assumption that an optimal larger plan can be built from optimal smaller plans. The lecture also keeps certain non-cheapest partial plans if they produce an **interesting order**: data sorted on an attribute needed later for `ORDER BY`, `GROUP BY`, or a join. Sorting costs work now, but may save more work later. So DP retains the best plan for each useful ordering as well as the best plan without a special ordering.


## 9. Oracle query optimization (PDF pages 39–45)

The lecture contrasts:

- **Rule-based optimization:** rank available access strategies and choose the highest-ranked one that can be used. This is the older/deprecated approach discussed in the slides.
- **Cost-based optimization:** estimate alternative plan costs from database statistics and choose a low-cost plan. The goal can emphasize throughput (overall work) or response time (how soon results begin arriving).

Oracle gathers statistics automatically in many cases, with histograms as a special case discussed in the slides. Basic statistics often assume uniform data distribution. A **histogram** records more information about how values are distributed, helping estimates when the data is skewed.

Oracle can also show execution plans, save plans, or accept hints. A hint influences the optimizer; it is not a substitute for checking whether the plan performs well on the actual data.


## 10. Worked comparison: why statistics matter

Suppose `Employee` has 10,000 rows and `department_id` has 100 distinct values, assumed evenly distributed. For:

```sql
SELECT * FROM Employee WHERE department_id = 17;
```

a rough estimate is `10,000 / 100 = 100` matching rows. If the table occupies 200 blocks, a full scan reads about 200 blocks. If there is a useful index, the DBMS can locate the matching rows without scanning every block; whether that is cheaper depends on index depth, clustering, and how many blocks hold the 100 matches.

If department 17 actually contains 4,000 rows, the uniform-distribution estimate is poor. Better statistics (such as a histogram) can help the optimizer avoid choosing a bad plan.


## Exam connection and revision questions

Query processing is the central focus of **Q2 in the 2023, 2024, and 2025 COMP0178 past papers**, as summarized in [`exam_guide.ipynb`](../exam_guide.ipynb). The papers ask you to translate a business question into SQL/relational algebra and reason about equivalent plans and costs. Past papers guide practice; they do not predict the next exam. The 2024 Q2 mark discrepancy is noted in the exam guide.

<details><summary>1. What do nTuples, bFactor, and nBlocks describe?</summary>

`nTuples` is row count; `bFactor` is tuples per block; `nBlocks` is the number of blocks required.
</details>

<details><summary>2. A table has 600 rows and an attribute has 30 distinct values. Estimate rows for one equality selection under uniformity.</summary>

`600 / 30 = 20` rows.
</details>

<details><summary>3. When is indexed nested-loop join attractive?</summary>

When an index exists on the inner relation’s join attribute, especially if the outer input is small enough that repeated indexed lookups are cheaper than scanning/sorting both relations.
</details>

<details><summary>4. Why keep an interesting-order plan in dynamic programming?</summary>

It may cost more now to sort, but the resulting order can make a later join, grouping, or ORDER BY cheaper.
</details>

<details><summary>5. What is the difference between materialization and pipelining?</summary>

Materialization writes an intermediate result to disk for later rereading; pipelining sends rows to the next operation directly.
</details>
